# NB 9 — The agent on a synthetic FHIR sandbox
**Goal:** move the toy tools of NB 2/NB 3 onto a real **FHIR** interface — the agent reads an INR and medications as FHIR resources and writes a flag back — while the *loop and the gate stay identical.* This is the bridge from teaching scaffold to the virtual patient environment.

It ships a **mock FHIR client** so it runs with no infrastructure. To use real data, generate patients with **Synthea**, load them into a **HAPI FHIR** server, and set `FHIR_BASE_URL` — the agent code doesn't change. (Runs in MOCK mode with no API key.)

In [1]:
import os, json, re

# =============================================================
# Model backend — works two ways:
#   1) MOCK (default): no API key needed. Returns scripted responses
#      so you can run the whole notebook and see the STRUCTURE.
#   2) REAL model: pip install openai, then either
#        - Cloud:  export OPENAI_API_KEY=sk-...        (uses OpenAI)
#        - Local open-weight (vLLM / LM Studio / Ollama):
#              export OPENAI_BASE_URL=http://localhost:8000/v1
#              export OPENAI_API_KEY=dummy
#              export MODEL=meta-llama/Llama-3.1-8B-Instruct   # your served model
# Everything below is model-agnostic: swap the model, keep the code.
# =============================================================
USE_MOCK = os.environ.get("OPENAI_API_KEY") is None
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

def chat(messages, temperature=0):
    """Return the assistant's text for a list of {role, content} messages."""
    if USE_MOCK:
        return _mock(messages, temperature)
    from openai import OpenAI
    client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL"))  # None -> api.openai.com
    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature, timeout=60)
    return r.choices[0].message.content

print("Backend:", "MOCK (no key found — scripted demo)" if USE_MOCK else f"REAL model = {MODEL}")

def _mock(messages, temperature=0):
    "Scripted ReAct actions for the FHIR-backed anticoagulant loop."
    hist = " ".join(m["content"] for m in messages if m["role"] != "system")
    if "get_latest_inr" not in hist:
        return json.dumps({"thought":"Read the latest INR via FHIR.","action":"get_latest_inr","action_input":{}})
    if "get_medications" not in hist:
        return json.dumps({"thought":"Check active medications.","action":"get_medications","action_input":{}})
    if "write_flag" not in hist:
        return json.dumps({"thought":"INR 4.2 is out of range — draft a dose-review flag for the clinician.",
                           "action":"write_flag","action_input":{"text":"INR 4.2 above range; request dose review + earlier follow-up."}})
    return json.dumps({"thought":"Flag written and awaiting review.","action":"final","action_input":{"summary":"Drafted dose-review flag; no order changed."}})

Backend: REAL model = openai/gpt-4o-mini


### A synthetic FHIR patient + a FHIR client
The mock client mirrors the real FHIR REST surface (Observation for the INR, MedicationRequest for meds, a Communication for the write-back). If `FHIR_BASE_URL` is set, the same methods hit a real HAPI FHIR server instead.

In [2]:
# A Synthea-style FHIR bundle (trimmed). LOINC 6301-6 = INR; RxNorm 855332 = warfarin 5 mg.
BUNDLE = {
 "Observation": [{"code":"6301-6","display":"INR","valueQuantity":{"value":4.2,"unit":"ratio"},
                  "referenceRange":"2.0-3.0","effectiveDateTime":"today"}],
 "MedicationRequest": [{"medication":"warfarin 5 mg oral tablet","rxnorm":"855332","status":"active"}],
 "Communication": [],   # write-backs land here
}
BASE = os.environ.get("FHIR_BASE_URL")   # e.g. http://localhost:8080/fhir  (real HAPI FHIR)

class FHIR:
    def get_latest_inr(self, **_):
        if BASE:
            import requests
            r = requests.get(f"{BASE}/Observation", params={"code":"http://loinc.org|6301-6",
                             "_sort":"-date","_count":1}, timeout=10).json()
            o = r["entry"][0]["resource"]
            return {"inr":o["valueQuantity"]["value"], "reference_range":"2.0-3.0"}
        o = BUNDLE["Observation"][0]
        return {"inr":o["valueQuantity"]["value"], "reference_range":o["referenceRange"]}
    def get_medications(self, **_):
        if BASE:
            import requests
            r = requests.get(f"{BASE}/MedicationRequest", params={"status":"active"}, timeout=10).json()
            return {"medications":[e["resource"]["medicationCodeableConcept"]["text"] for e in r.get("entry",[])]}
        return {"medications":[m["medication"] for m in BUNDLE["MedicationRequest"]]}
    def write_flag(self, **a):
        text = next((a[k] for k in ("text","reason","summary","note","content","flag") if a.get(k)),
                    next((v for v in a.values() if isinstance(v,str) and v.strip()), ""))
        if BASE:
            import requests
            payload={"resourceType":"Communication","status":"preparation","payload":[{"contentString":text}]}
            requests.post(f"{BASE}/Communication", json=payload, timeout=10)
        BUNDLE["Communication"].append({"status":"preparation","payload":text})
        return {"written":text}

fhir = FHIR()
print("FHIR backend:", "REAL HAPI @ "+BASE if BASE else "MOCK (in-notebook synthetic bundle)")
print("Sanity read ->", fhir.get_latest_inr())

FHIR backend: MOCK (in-notebook synthetic bundle)
Sanity read -> {'inr': 4.2, 'reference_range': '2.0-3.0'}


### Tools wrap FHIR calls; the write-back is gated
Identical loop to NB 2, identical gate to NB 3 — only the tool bodies changed from a dict to FHIR calls.

In [3]:
TOOLS = {"get_latest_inr":fhir.get_latest_inr, "get_medications":fhir.get_medications, "write_flag":fhir.write_flag}
GATED = {"write_flag"}                      # writing to the record is consequential
AUTO  = os.environ.get("GATE_AUTO", "approve")   # demo default: approve; set 'deny' to see it blocked
def approves(a, args):
    print(f"   >> GATE: {a}({args}) — approve? [{AUTO}]"); return AUTO.lower().startswith("a")

SYSTEM = ("You are a clinical follow-up assistant working over FHIR. Output ONLY JSON "
          '{"thought","action","action_input"}. Actions: '+str(list(TOOLS)+["final"])+
          ". Read the INR and meds; if the INR is out of range, write_flag a dose-review request. Change no order.")
def parse(txt):
    t = re.sub(r"^```[a-z]*|```$","",txt.strip(),flags=re.M).strip()
    try:    return json.loads(t)
    except Exception:
        m = re.search(r"\{.*\}", t, re.S); return json.loads(m.group(0))

msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":"Handle the anticoagulant follow-up."}]
for step in range(6):
    act=parse(chat(msgs)); print(f"[{step}] {act.get('thought','')}")
    a=act.get("action","final")
    if a=="final": print("    DONE:", act.get("action_input",{}).get("summary","")); break
    if a not in TOOLS: print(f"    (unknown action {a!r}; stopping)"); break
    args=act.get("action_input",{})
    obs = TOOLS[a](**args) if not (a in GATED and not approves(a,args)) else {"blocked":"human denied"}
    print("    OBS:", obs)
    msgs += [{"role":"assistant","content":json.dumps(act)},
             {"role":"user","content":f"[{a}] observation: {json.dumps(obs)}"}]

[0] I need to check the latest INR and medications to ensure the patient is within the therapeutic range for anticoagulation.
    OBS: {'inr': 4.2, 'reference_range': '2.0-3.0'}


[1] The INR of 4.2 is out of the therapeutic range (2.0-3.0), indicating a potential risk for bleeding. I need to write a flag for a dose-review request.
   >> GATE: write_flag({'reason': 'INR out of range', 'inr_value': 4.2}) — approve? [approve]
    OBS: {'written': 'INR out of range'}


[2] The dose-review request has been successfully written due to the elevated INR. Now, I will check the patient's medications to ensure there are no contraindications or interactions.
    OBS: {'medications': ['warfarin 5 mg oral tablet']}


[3] The patient is on warfarin 5 mg, which is consistent with the elevated INR. No changes to the order are needed at this time. The follow-up is complete.
    DONE: 


### Read back what we wrote

In [4]:
print("Communications on the record:", BUNDLE["Communication"])

Communications on the record: [{'status': 'preparation', 'payload': 'INR out of range'}]


### Takeaway
The agent logic — perceive, plan, act, gated write — did not change; only the *environment* did, from a Python dict to FHIR resources. That is the whole point of building on standards: the same agent you prototyped on a toy runs against a synthetic Synthea patient today and a governed EHR sandbox tomorrow, unchanged. What Synthea gives you is the patient **state**; what this notebook adds is the **read/write interface**; the missing piece the paper flags is the **dynamics** (an action changing future state), which you build on top.

*To go live:* `git clone` Synthea, generate a population (`./run_synthea -p 100`), load the FHIR bundles into a HAPI FHIR server, then `export FHIR_BASE_URL=http://localhost:8080/fhir` and re-run — no code change.